# 02 — Global transmitted HIV drug resistance choropleth

This notebook maps descriptive, sample-size-weighted country estimates from eligible **single-country** studies in the 2015 published snapshot.

The map is an evidence-synthesis teaching example, **not a current national surveillance estimate**.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA = Path("data")
summary_path = DATA / "hivdb_country_summary_2015.csv"
studies_path = DATA / "hivdb_surveillance_studies_2015.csv"
study_countries_path = DATA / "hivdb_study_countries_2015.csv"
metadata_path = DATA / "hivdb_2015_metadata.json"

required = [summary_path, studies_path, study_countries_path, metadata_path]
missing = [p.name for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Generated data are missing: " + ", ".join(missing) +
        ". In the deployed GitHub Pages site these files are generated during the build. "
        "For local use, run: python scripts/fetch_plos_2015.py --output-dir content/data"
    )

summary = pd.read_csv(summary_path)
studies = pd.read_csv(studies_path)
study_countries = pd.read_csv(study_countries_path)
metadata = json.loads(metadata_path.read_text())


In [ ]:
import plotly.express as px

OUTCOMES = {
    "Overall TDR": "tdr_overall_pct",
    "NRTI TDR": "tdr_nrti_pct",
    "NNRTI TDR": "tdr_nnrti_pct",
    "PI TDR": "tdr_pi_pct",
}

def make_map(label="Overall TDR"):
    col = OUTCOMES[label]
    if col not in summary.columns:
        raise KeyError(f"{col} was not available in this source snapshot")
    plot_data = summary.loc[summary[col].notna()].copy()
    fig = px.choropleth(
        plot_data,
        locations="iso3",
        color=col,
        hover_name="country",
        hover_data={
            "iso3": True,
            "n_studies_total": True,
            "n_studies_single_country": True,
            "n_participants_weighted": ":,.0f",
            col: ":.1f",
        },
        color_continuous_scale="YlOrRd",
        range_color=(0, max(20, float(plot_data[col].quantile(0.95)))) if not plot_data.empty else None,
        labels={col: f"{label} (%)"},
        title=f"{label}: 2015 published HIVDB surveillance snapshot",
    )
    fig.update_geos(showframe=False, showcoastlines=True, projection_type="natural earth")
    fig.update_layout(margin=dict(l=0, r=0, t=60, b=0))
    return fig

make_map("Overall TDR").show()

## Change the resistance class

Try one of `"NRTI TDR"`, `"NNRTI TDR"`, or `"PI TDR"`. Missing source values remain uncolored rather than being interpreted as zero.

In [ ]:
make_map("NNRTI TDR").show()

## Map evidence density alongside prevalence

A prevalence map without an evidence-density map can imply false precision. The next map shows how many source studies are associated with each country.

In [ ]:
fig = px.choropleth(
    summary,
    locations="iso3",
    color="n_studies_total",
    hover_name="country",
    hover_data=["n_studies_total", "n_studies_single_country", "has_weighted_estimate"],
    color_continuous_scale="Blues",
    title="Number of source studies represented by country",
)
fig.update_geos(showframe=False, showcoastlines=True, projection_type="natural earth")
fig.update_layout(margin=dict(l=0, r=0, t=60, b=0))
fig.show()

## Interpretation checklist

- Is the estimate supported by one study or many?
- Are those studies old or recent relative to the question?
- Is the sample nationally representative?
- Were participants newly infected or chronically infected but ART-naïve?
- Are drug classes and SDRM definitions comparable across the studies?
- Could publication geography be mistaken for epidemiologic geography?